# Notebook 03 — Cross-Correlation of Waveforms

**Pipeline stage:** B  
**Input:** `02-data/A_ID/waves_Y1Y2.h5` (written by Notebook 02)  
**Output:** `02-data/B_CC/cc_Y1Y2.h5`  
**Translation of:** `B_CC_all.m`

---

## What is cross-correlation and why do we use it?

Cross-correlation is a mathematical way to measure how *similar* two signals are to each other,
and by how much one would need to be shifted in time to best align with the other.

For seismology, the idea is simple: **if two earthquakes occurred in nearly the same location**,
their waveforms recorded at the same seismometer should look almost identical — same shape,
same timing, just maybe scaled differently.  The P-wave first motion (the very first wiggle of
the waveform) carries the polarity information (up = compression, down = dilation) that we
need for focal mechanism inversion.

The cross-correlation score (CC) between two waveforms ranges from **-1 to +1**:
- **CC ≈ +1**: waveforms look nearly identical → events very likely have the **same polarity** at this station.
- **CC ≈ -1**: waveforms are mirror images → events very likely have **opposite polarity**.
- **CC ≈ 0**: waveforms are unrelated.

We use a threshold of `CC_THRESHOLD = 0.70`.  Only pairs above this threshold are stored,
because weaker correlations are not reliable enough to use in the SVD clustering step.

### Why correlate a short window?

We only correlate a **short 0.20-second window** around the P-wave onset (the `CC_WIN_SHORT`
window, starting 0.07 s before the pick).  Using a long window would drag in surface-reflected
phases and coda, which can accidentally make dissimilar events look similar.

### What do we do with the CC pairs?

In the next notebook (04), we will build a *similarity graph* where each event is a node and
each high-CC pair is an edge with weight equal to the CC value.  The leading eigenvector of that
graph (from Singular Value Decomposition, SVD) gives a **polarity score** for every event at
every station.  The *sign* of the score tells us polarity direction; the *magnitude* tells us
confidence.

### Scale challenge: 54,971 events

With ~55 k events, a naive all-pairs loop would require checking
55000 × 54999 / 2 ≈ **1.5 billion pairs**.  Even at 1 µs per pair that is 25 minutes per station,
times 21 stations.  Instead we use **batched matrix multiplication**: normalise every waveform
to unit length, then compute dot products in blocks using NumPy's optimised BLAS routines.
This reduces the per-station time to a few minutes on a modern laptop.


In [ ]:
import sys
import time
import traceback
from pathlib import Path

import numpy as np
import h5py
import matplotlib
matplotlib.use('Agg')          # non-interactive backend — safe for long runs
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy.signal import correlate, correlation_lags

print('Imports OK')
print(f'NumPy  {np.__version__}')
print(f'h5py   {h5py.__version__}')

In [ ]:
# Add the python/ directory to the path so we can import pipeline_config
PYTHON_DIR = Path(__file__).resolve().parent if '__file__' in dir() else Path.cwd()
if str(PYTHON_DIR) not in sys.path:
    sys.path.insert(0, str(PYTHON_DIR))

from pipeline_config import (
    WAVES_H5, CC_H5, EMAIL_PLOTS_DIR,
    STATIONS_ALL,
    FS, WIN_START, N_SAMPLES,
    CC_THRESHOLD, CC_LAG_AHEAD, CC_WIN_SHORT,
)
from email_utils import step_email, error_email

# ── Derived CC window indices ──────────────────────────────────────────────
# P pick sits at sample index  abs(WIN_START) * FS  =  0.32 * 200  =  64
# We start the CC window CC_LAG_AHEAD seconds BEFORE the pick:
#   cc_start = (abs(WIN_START) - CC_LAG_AHEAD) * FS  =  (0.32 - 0.10) * 200  = 44
# but B_CC_all.m uses a slightly different convention — it starts 0.25 s before
# the pick, giving  (0.32 - 0.25) * 200 = 14.
CC_START_IDX = int((abs(WIN_START) - 0.25) * FS)    # = 14
CC_LEN       = int(CC_WIN_SHORT * FS)                # = 40 samples  (0.20 s)

print(f'Waveform HDF5  : {WAVES_H5}')
print(f'CC output HDF5 : {CC_H5}')
print(f'Stations       : {len(STATIONS_ALL)}')
print(f'CC window      : samples {CC_START_IDX} – {CC_START_IDX + CC_LEN - 1}')
print(f'CC threshold   : {CC_THRESHOLD}')

# Make output directory if needed
CC_H5.parent.mkdir(parents=True, exist_ok=True)
EMAIL_PLOTS_DIR.mkdir(parents=True, exist_ok=True)

## How the CC algorithm works — step by step

### Step 1 — Load the CC snippet for all events at one station

For each station we load **all** waveforms from the HDF5 file.  The full waveform is 264 samples
long (from −0.32 s to +1.00 s relative to the P pick at 200 Hz).  We extract only the short
CC window: samples `CC_START_IDX` to `CC_START_IDX + CC_LEN`.

### Step 2 — Normalise to unit L2 norm

Before computing dot products, each snippet is divided by its own L2 norm:

```
w_norm[i] = w[i] / ||w[i]||
```

After this step, the dot product of any two normalised snippets equals their
**Pearson correlation coefficient** (zero-lag cross-correlation).

Waveforms with near-zero energy (the station did not record this event well) are
left as zero vectors and will produce CC = 0 with everything, so they are harmlessly
excluded.

### Step 3 — Batched matrix multiplication

We process the events in blocks of `batch_size = 500` rows at a time:

```
cc_block[i, j] = W[batch_start + i, :] · W[j, :]   (dot product)
```

This single `@` operator maps to a highly optimised BLAS `DGEMM` call inside NumPy,
which is **much** faster than a Python loop.

The result is a (batch_size × N) matrix of CC values.  We only keep entries where
`cc > CC_THRESHOLD` and the pair is in the **upper triangle** (`i < j`) to avoid
storing each pair twice.

### Step 4 — Save to HDF5

Results are stored as a 2-D array of shape **(M, 3)** in the HDF5 file under the group
named after the station.  Each row is `[event_index_1, event_index_2, cc_value]`.
We store *indices* into the `event_ids` array (not the IDs themselves) for compact storage.


In [ ]:
def compute_cc_matrix_batched(
    W: np.ndarray,
    threshold: float = CC_THRESHOLD,
    batch_size: int = 500,
) -> np.ndarray:
    """
    Compute all-pairs zero-lag cross-correlation for a waveform matrix.

    Parameters
    ----------
    W          : (N, n_samples) float32 array of unit-normalised waveform snippets.
    threshold  : only keep pairs with CC > threshold.
    batch_size : number of rows processed per dot-product call.

    Returns
    -------
    pairs : (M, 3) float32 array  — columns [idx1, idx2, cc_value]
            upper-triangle only (idx1 < idx2).
    """
    N = len(W)
    all_pairs = []

    for start in range(0, N, batch_size):
        end   = min(start + batch_size, N)
        chunk = W[start:end]          # (batch, n_samples)

        # Dot product gives zero-lag CC because W is unit-normalised
        cc_block = chunk @ W.T        # (batch, N)

        # Find entries above threshold — only upper triangle (start+r < c)
        rows, cols = np.where(cc_block > threshold)
        for r, c in zip(rows, cols):
            global_r = start + r
            if global_r < c:          # upper triangle only
                all_pairs.append((global_r, c, cc_block[r, c]))

    if len(all_pairs) == 0:
        return np.empty((0, 3), dtype=np.float32)
    return np.array(all_pairs, dtype=np.float32)


def compute_cc_for_station(
    station: str,
    waves_h5_path: Path,
    threshold: float = CC_THRESHOLD,
    cc_start: int = CC_START_IDX,
    cc_len: int = CC_LEN,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Load waveforms for one station, normalise, and compute high-CC pairs.

    Parameters
    ----------
    station       : station short name, e.g. 'AS1' or '01A'.
    waves_h5_path : path to waves_Y1Y2.h5.
    threshold     : CC threshold.
    cc_start      : first sample index of the CC window.
    cc_len        : number of samples in the CC window.

    Returns
    -------
    event_ids : (N,) int64 array — event IDs.
    pairs     : (M, 3) float32 array — [idx1, idx2, cc_value].
    """
    with h5py.File(waves_h5_path, 'r') as f:
        event_ids = f['event_ids'][:]                     # (N,) int64

        if f'waveforms/{station}' not in f:
            print(f'  [skip] {station} — no waveform group in HDF5')
            return event_ids, np.empty((0, 3), dtype=np.float32)

        waves = f[f'waveforms/{station}'][:]              # (N, 264) float32

    N = len(waves)

    # ── Extract the short CC window ───────────────────────────────────────
    snippets = waves[:, cc_start : cc_start + cc_len].astype(np.float32)  # (N, 40)

    # ── Unit-normalise each row ───────────────────────────────────────────
    norms = np.linalg.norm(snippets, axis=1, keepdims=True)   # (N, 1)
    # Avoid division by zero for flat traces (station not recording this event)
    valid = (norms[:, 0] > 1e-10)
    norms[~valid] = 1.0            # flat traces → zero vector after division
    W = snippets / norms           # (N, 40) unit-normalised
    W[~valid] = 0.0                # explicitly zero out flat traces

    n_valid = valid.sum()
    print(f'  {station}: {N} events, {n_valid} with valid waveforms')

    # ── Batched CC computation ────────────────────────────────────────────
    pairs = compute_cc_matrix_batched(W, threshold=threshold)

    return event_ids, pairs


print('Functions defined.')

In [ ]:
# ── Main loop: compute CC for every station and write to HDF5 ────────────

t0 = time.time()
station_stats = {}   # station → number of high-CC pairs found

try:
    with h5py.File(CC_H5, 'w') as out_f:
        # Store the global event_ids once (same order for all stations)
        with h5py.File(WAVES_H5, 'r') as wf:
            event_ids_global = wf['event_ids'][:]
        out_f.create_dataset('event_ids', data=event_ids_global, compression='gzip')

        N_events = len(event_ids_global)
        print(f'Total events: {N_events:,}')
        print(f'Theoretical max pairs: {N_events * (N_events - 1) // 2:,}')
        print()

        for sta in tqdm(STATIONS_ALL, desc='Stations'):
            t_sta = time.time()
            _, pairs = compute_cc_for_station(sta, WAVES_H5)
            elapsed = time.time() - t_sta

            n_pairs = len(pairs)
            station_stats[sta] = n_pairs
            print(f'  → {n_pairs:,} pairs above {CC_THRESHOLD}  ({elapsed:.1f} s)')

            if n_pairs > 0:
                out_f.create_dataset(
                    sta, data=pairs,
                    compression='gzip', compression_opts=4
                )
            else:
                # Create empty dataset so downstream code can always find the key
                out_f.create_dataset(sta, data=np.empty((0, 3), dtype=np.float32))

    total_elapsed = time.time() - t0
    total_pairs   = sum(station_stats.values())
    print(f'\nDone. Total pairs across all stations: {total_pairs:,}')
    print(f'Total time: {total_elapsed/60:.1f} min')
    print(f'Output: {CC_H5}')

except Exception:
    error_email(3, 'Cross-Correlation', traceback.format_exc())
    raise

In [ ]:
# ── Diagnostic plots ─────────────────────────────────────────────────────

saved_plots = []

# ── Plot 1: pairs-per-station bar chart ──────────────────────────────────
fig1, ax1 = plt.subplots(figsize=(12, 5))
stations = list(station_stats.keys())
counts   = [station_stats[s] for s in stations]

bars = ax1.bar(stations, counts, color='steelblue', edgecolor='white', linewidth=0.5)
ax1.set_xlabel('Station', fontsize=12)
ax1.set_ylabel('High-CC pairs (CC > {:.2f})'.format(CC_THRESHOLD), fontsize=12)
ax1.set_title('Cross-Correlation: high-CC pairs per station\n'
              f'(CC_WIN_SHORT = {CC_WIN_SHORT} s, {CC_LEN} samples)', fontsize=13)
ax1.tick_params(axis='x', rotation=45)

# Annotate each bar with its count
for bar, cnt in zip(bars, counts):
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(counts) * 0.01,
        f'{cnt:,}', ha='center', va='bottom', fontsize=7, rotation=90
    )

fig1.tight_layout()
p1 = EMAIL_PLOTS_DIR / 'step03_pairs_per_station.png'
fig1.savefig(p1, dpi=150)
plt.close(fig1)
saved_plots.append(p1)
print(f'Saved: {p1}')


# ── Plot 2: CC value histogram (sample one station with the most pairs) ──
best_sta = max(station_stats, key=station_stats.get)

with h5py.File(CC_H5, 'r') as f:
    sample_pairs = f[best_sta][:]   # (M, 3)

if len(sample_pairs) > 0:
    cc_vals = sample_pairs[:, 2]

    fig2, ax2 = plt.subplots(figsize=(8, 4))
    ax2.hist(cc_vals, bins=60, range=(CC_THRESHOLD, 1.0),
             color='darkorange', edgecolor='white', linewidth=0.4)
    ax2.axvline(CC_THRESHOLD, color='red', linestyle='--',
                label=f'threshold = {CC_THRESHOLD}')
    ax2.set_xlabel('Zero-lag CC value', fontsize=12)
    ax2.set_ylabel('Number of pairs', fontsize=12)
    ax2.set_title(f'Distribution of CC values — station {best_sta} '
                  f'({len(cc_vals):,} pairs above threshold)', fontsize=12)
    ax2.legend()
    fig2.tight_layout()
    p2 = EMAIL_PLOTS_DIR / 'step03_cc_histogram.png'
    fig2.savefig(p2, dpi=150)
    plt.close(fig2)
    saved_plots.append(p2)
    print(f'Saved: {p2}')


# ── Plot 3: Summary heat-map — pairs per station as a colored grid ───────
# (useful to spot stations that recorded very few events)
fig3, ax3 = plt.subplots(figsize=(10, 2.5))
vals_arr = np.array(counts, dtype=float).reshape(1, -1)
im = ax3.imshow(vals_arr, aspect='auto', cmap='YlOrRd')
ax3.set_xticks(range(len(stations)))
ax3.set_xticklabels(stations, rotation=45, ha='right', fontsize=9)
ax3.set_yticks([])
ax3.set_title('High-CC pairs per station (colour = count)', fontsize=11)
plt.colorbar(im, ax=ax3, orientation='horizontal', pad=0.4, label='pair count')
fig3.tight_layout()
p3 = EMAIL_PLOTS_DIR / 'step03_cc_heatmap.png'
fig3.savefig(p3, dpi=150)
plt.close(fig3)
saved_plots.append(p3)
print(f'Saved: {p3}')

print('\nAll plots saved.')

In [ ]:
# ── Send completion email ─────────────────────────────────────────────────

stats_dict = {
    'Events processed'           : f'{N_events:,}',
    'Stations'                   : len(STATIONS_ALL),
    'CC threshold'               : CC_THRESHOLD,
    'CC window length (samples)' : CC_LEN,
    'Total high-CC pairs'        : f'{total_pairs:,}',
    'Best station (most pairs)'  : f'{best_sta}  ({station_stats[best_sta]:,} pairs)',
    'Wall-clock time'            : f'{total_elapsed/60:.1f} min',
    'Output file'                : str(CC_H5),
}
# Add per-station pair counts
for sta, cnt in station_stats.items():
    stats_dict[f'  {sta} pairs'] = f'{cnt:,}'

step_email(
    step_num=3,
    step_name='Cross-Correlation',
    stats=stats_dict,
    plots=saved_plots,
    review_required=False,
)

print('Step 03 complete — ready for Notebook 04 (SVD clustering).')